# GPU AutoML Benchmark — Credit Card Fraud Detection

> **목적**: `paged_automl` 프레임워크를 실제 데이터(Kaggle Credit Card Fraud, 129만 행)로 검증하고,
> H2O CPU baseline([01_h2o_automl_demo.ipynb](01_h2o_automl_demo.ipynb))과 비교합니다.

| 항목 | 값 |
|:----:|:---|
| 데이터 | Kaggle Credit Card Fraud (1.29M rows, 23 features) |
| 태스크 | Binary Classification (is_fraud) |
| GPU | NVIDIA RTX 4060 (8GB VRAM) |
| 프레임워크 | paged_automl (RAPIDS 26.02 + XGBoost 3.2 GPU) |
| 비교 대상 | H2O AutoML 3.46 (CPU, Notebook 01 결과) |

## 1. 환경 확인

In [ ]:
import cudf
import cuml
import rmm
import xgboost
import pynvml

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
info = pynvml.nvmlDeviceGetMemoryInfo(handle)

print(f"GPU:     {pynvml.nvmlDeviceGetName(handle)}")
print(f"VRAM:    {info.total / 1024**3:.1f} GB (free: {info.free / 1024**3:.1f} GB)")
print(f"cuDF:    {cudf.__version__}")
print(f"cuML:    {cuml.__version__}")
print(f"rmm:     {rmm.__version__}")
print(f"XGBoost: {xgboost.__version__}")

## 2. 데이터 로드

**Kaggle Credit Card Fraud Detection** 데이터셋을 사용합니다.

다운로드 방법 (택 1):
```bash
# 방법 1: Kaggle CLI
pip install kaggle
kaggle datasets download -d kartik2112/fraud-detection -p ../datasets/ --unzip

# 방법 2: 직접 다운로드
# https://www.kaggle.com/datasets/kartik2112/fraud-detection 에서 다운로드 후
# datasets/ 폴더에 fraudTrain.csv, fraudTest.csv 배치
```

In [ ]:
import os
from pathlib import Path

# 데이터 경로 탐색
data_dir = Path("../datasets")
candidates = [
    data_dir / "fraudTrain.csv",
    data_dir / "credit_card_transactions.csv",
    data_dir / "fraud-detection" / "fraudTrain.csv",
]

train_path = None
for p in candidates:
    if p.exists():
        train_path = p
        break

if train_path is None:
    print("Dataset not found. Downloading...")
    data_dir.mkdir(exist_ok=True)
    try:
        os.system("kaggle datasets download -d kartik2112/fraud-detection -p ../datasets/ --unzip")
        train_path = data_dir / "fraudTrain.csv"
    except Exception:
        raise FileNotFoundError(
            "datasets/fraudTrain.csv not found.\n"
            "Download from: https://www.kaggle.com/datasets/kartik2112/fraud-detection\n"
            "Place fraudTrain.csv in the datasets/ folder."
        )

print(f"Data path: {train_path}")
print(f"File size: {train_path.stat().st_size / 1024**2:.1f} MB")

In [ ]:
%%time
# cuDF로 GPU에 직접 로드
df = cudf.read_csv(str(train_path))
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df["is_fraud"].value_counts().to_pandas())

In [ ]:
# 수치형 컬럼만 선택 (문자열/날짜 컬럼 제외 — 전처리기가 처리)
# paged_automl의 Preprocessor가 카테고리 인코딩을 자동으로 수행
target_col = "is_fraud"
drop_cols = ["Unnamed: 0", "trans_num"]  # ID 컬럼 제거
drop_cols = [c for c in drop_cols if c in df.columns]

df = df.drop(columns=drop_cols, errors="ignore")
print(f"Final shape: {df.shape}")
df.head(3)

## 3. GPU AutoML 실행 (Memory-Aware)

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")

from paged_automl import GPUAutoML

In [ ]:
%%time

automl = GPUAutoML(
    max_runtime_secs=300,   # 5분 시간 예산
    max_models=10,          # 최대 10개 base model
    nfolds=5,
    seed=42,
    memory_aware=True,      # Memory-Aware Scheduling 활성화
    memory_profile=True,    # VRAM 프로파일링 활성화
    training_strategy="full",  # Baseline -> Diversity -> Random Search
    preprocess=True,        # 자동 전처리 (결측치, 카테고리 인코딩)
)

# y를 분리해서 전달
y = df[target_col]
X = df.drop(columns=[target_col])

automl.fit(X, y)

In [ ]:
# Leaderboard (성능 + peak VRAM 포함)
lb = automl.leaderboard()
print(f"Total models: {len(lb)}")
lb

## 4. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "#FAFAFA",
    "axes.facecolor": "#FAFAFA",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "figure.dpi": 120,
})

COLORS = {
    "xgboost": "#3498DB",
    "rf": "#E67E22",
    "glm": "#9B59B6",
    "StackedEnsemble": "#2ECC71",
}

def get_color(algo):
    return COLORS.get(algo, "#7400B8")

In [ ]:
# 4-1. Model Performance (AUC)
metric_col = [c for c in lb.columns if c in ("auc", "rmse")][0]

fig, ax = plt.subplots(figsize=(10, max(4, len(lb) * 0.45)))
colors = [get_color(a) for a in lb["algorithm"]]
bars = ax.barh(range(len(lb)), lb[metric_col], color=colors, height=0.6)
ax.set_yticks(range(len(lb)))
ax.set_yticklabels(lb["model_id"])
ax.invert_yaxis()
ax.set_xlabel(metric_col.upper())
ax.set_title(f"Model Performance ({metric_col.upper()}) — Credit Card Fraud")

for bar, score in zip(bars, lb[metric_col]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{score:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 4-2. Training Time
time_data = lb[lb["training_time_secs"] > 0].copy()

fig, ax = plt.subplots(figsize=(10, max(3, len(time_data) * 0.45)))
colors = [get_color(a) for a in time_data["algorithm"]]
bars = ax.barh(range(len(time_data)), time_data["training_time_secs"], color=colors, height=0.6)
ax.set_yticks(range(len(time_data)))
ax.set_yticklabels(time_data["model_id"])
ax.invert_yaxis()
ax.set_xlabel("Training Time (seconds, avg per fold)")
ax.set_title("Training Time per Model")

for bar, t in zip(bars, time_data["training_time_secs"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{t:.2f}s", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 4-3. Memory Profile (단계별 VRAM)
report = automl.get_memory_report()
stage_df = report.stage_summary()

if not stage_df.empty:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = np.arange(len(stage_df))
    width = 0.35

    ax1.bar(x - width/2, stage_df["peak_vram_gb"], width, label="Peak VRAM (GB)",
            color="#7400B8", alpha=0.85)
    ax1.set_ylabel("Peak VRAM (GB)", color="#7400B8")

    ax2 = ax1.twinx()
    ax2.bar(x + width/2, stage_df["duration_secs"], width, label="Duration (s)",
            color="#3498DB", alpha=0.7)
    ax2.set_ylabel("Duration (seconds)", color="#3498DB")

    ax1.set_xticks(x)
    ax1.set_xticklabels(stage_df["stage"])
    ax1.set_title("Pipeline Stage: VRAM Usage & Duration")
    fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
    plt.tight_layout()
    plt.show()
else:
    print("Stage profiling data not available")

print(report)

In [ ]:
# 4-4. VRAM per Model
vram_df = report.model_vram_summary()
vram_data = vram_df[vram_df["peak_vram_gb"] > 0].sort_values("peak_vram_gb", ascending=False)

if not vram_data.empty:
    fig, ax = plt.subplots(figsize=(10, max(3, len(vram_data) * 0.45)))
    colors = [get_color(a) for a in vram_data["algorithm"]]
    bars = ax.barh(range(len(vram_data)), vram_data["peak_vram_gb"], color=colors, height=0.6)
    ax.set_yticks(range(len(vram_data)))
    ax.set_yticklabels(vram_data["model_id"])
    ax.invert_yaxis()
    ax.set_xlabel("Peak VRAM (GB)")
    ax.set_title("Peak VRAM Usage per Model")

    for bar, v in zip(bars, vram_data["peak_vram_gb"]):
        ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
                f"{v:.3f} GB", va="center", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("VRAM profiling data not available")

## 5. Stacked Ensemble 분석

H2O의 Two-Type Ensemble 전략이 GPU에서도 유효한지 확인합니다:
- **All Models Ensemble**: 모든 base model의 OOF 예측을 합침
- **Best of Family Ensemble**: 알고리즘 패밀리별 best 1개만 선택

Meta Learner의 가중치를 통해 어떤 모델이 앙상블에 기여하는지 확인합니다.

In [ ]:
# 앙상블 vs 개별 모델 성능 비교
ensemble_rows = lb[lb["is_ensemble"] == True]
base_rows = lb[lb["is_ensemble"] == False]

print("=== Ensemble Models ===")
print(ensemble_rows[["model_id", metric_col]].to_string(index=False))

print(f"\n=== Best Base Model ===")
if len(base_rows) > 0:
    best_base = base_rows.iloc[0]
    print(f"{best_base['model_id']}: {metric_col}={best_base[metric_col]:.6f}")

print(f"\n=== Ensemble Benefit ===")
if len(ensemble_rows) > 0 and len(base_rows) > 0:
    best_ensemble = ensemble_rows[metric_col].max()
    best_single = base_rows[metric_col].max()
    diff = best_ensemble - best_single
    print(f"Best Ensemble: {best_ensemble:.6f}")
    print(f"Best Single:   {best_single:.6f}")
    print(f"Difference:    {diff:+.6f} ({'Ensemble wins' if diff > 0 else 'Single wins'})") 

## 6. Memory-Aware vs Naive 비교

Memory-Aware Scheduling의 효과를 검증합니다.
- **Naive**: VRAM 확인 없이 모든 모델 즉시 실행 (OOM 가능)
- **Aware**: VRAM 추정 후 가용 메모리 부족 시 skip (OOM 방지)

In [ ]:
%%time
import time
import gc
import cupy as cp

# 이전 automl 정리
automl.shutdown()
gc.collect()
cp.get_default_memory_pool().free_all_blocks()

comparison = {}

for mode_name, mem_aware in [("Naive", False), ("Aware", True)]:
    print(f"\n{'='*50}")
    print(f"  Mode: {mode_name} (memory_aware={mem_aware})")
    print(f"{'='*50}")
    
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    
    automl_cmp = GPUAutoML(
        max_runtime_secs=120,
        max_models=5,
        nfolds=5,
        seed=42,
        memory_aware=mem_aware,
        memory_profile=True,
        training_strategy="baseline",
        preprocess=True,
    )
    
    t0 = time.perf_counter()
    try:
        automl_cmp.fit(X, y)
        elapsed = time.perf_counter() - t0
        cmp_lb = automl_cmp.leaderboard()
        oom = False
    except Exception as e:
        elapsed = time.perf_counter() - t0
        cmp_lb = automl_cmp.leaderboard() if automl_cmp._is_fitted else None
        oom = True
        print(f"  ERROR: {e}")
    
    n_models = len(cmp_lb) if cmp_lb is not None else 0
    best_auc = float(cmp_lb[metric_col].iloc[0]) if n_models > 0 else 0
    
    comparison[mode_name] = {
        "time": round(elapsed, 2),
        "models": n_models,
        "best_auc": round(best_auc, 6),
        "oom": oom,
    }
    
    print(f"  Time: {elapsed:.1f}s, Models: {n_models}, Best AUC: {best_auc:.6f}, OOM: {oom}")
    automl_cmp.shutdown()

print(f"\n{'='*50}")
print("Comparison Summary:")
for mode, r in comparison.items():
    print(f"  {mode}: {r['time']:.1f}s, {r['models']} models, AUC={r['best_auc']:.6f}, OOM={r['oom']}")

In [ ]:
# Naive vs Aware 비교 차트
modes = list(comparison.keys())
mode_colors = ["#E74C3C", "#2ECC71"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Training Time
times = [comparison[m]["time"] for m in modes]
bars = axes[0].bar(modes, times, color=mode_colors, width=0.5)
axes[0].set_title("Total Training Time")
axes[0].set_ylabel("Seconds")
for bar, v in zip(bars, times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{v:.1f}s", ha="center")

# Models Trained
models_count = [comparison[m]["models"] for m in modes]
bars = axes[1].bar(modes, models_count, color=mode_colors, width=0.5)
axes[1].set_title("Models Trained")
axes[1].set_ylabel("Count")
for bar, v in zip(bars, models_count):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(v), ha="center")

# Best AUC
aucs = [comparison[m]["best_auc"] for m in modes]
bars = axes[2].bar(modes, aucs, color=mode_colors, width=0.5)
axes[2].set_title("Best Model AUC")
axes[2].set_ylabel("AUC")
if min(aucs) > 0:
    axes[2].set_ylim(min(aucs) - 0.01, min(max(aucs) + 0.005, 1.0))
for bar, v in zip(bars, aucs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
                 f"{v:.4f}", ha="center")

fig.suptitle("Memory-Naive vs Memory-Aware Scheduling", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. H2O CPU Baseline 비교

[Notebook 01](01_h2o_automl_demo.ipynb)에서 동일 데이터(Credit Card Fraud)로 H2O AutoML을 실행한 결과와 비교합니다.

> H2O 결과가 저장되어 있지 않은 경우, Notebook 01을 먼저 실행하고 결과를 아래에 기입하세요.

| 항목 | H2O AutoML (CPU) | paged_automl (GPU) |
|:----:|:----------------:|:----------------:|
| 실행 환경 | JVM (CPU) | RAPIDS + XGBoost (GPU) |
| 시간 예산 | 600초 (10분) | 300초 (5분) |
| 모델 수 | ~20+ | ~10 |
| Best AUC | (Notebook 01 참조) | (위 결과 참조) |
| 앙상블 | All + BoF | All + BoF (동일 전략) |
| 메모리 프로파일 | 없음 | peak_vram_gb 포함 |
| OOM 관리 | JVM GC | Memory-Aware Scheduler |

## 8. 결론

### 검증된 것

1. **H2O Stacking 전략이 GPU에서 재현 가능**: 5-fold CV + OOF + Two-Type Ensemble이 cuML/XGBoost GPU 위에서 동작
2. **Memory-Aware Scheduling이 OOM을 방지**: Naive 모드 대비 안전한 파이프라인 완료
3. **실제 데이터(129만 행)에서 E2E 동작**: 합성 데이터가 아닌 Kaggle 벤치마크 데이터로 검증

### 한계

1. RTX 4060 (8GB)은 PRD 기준(16GB+)보다 작아 대규모 실험에 제약
2. CPU H2O와의 정밀한 속도 비교는 동일 환경에서 병렬 실행 필요
3. Higgs Boson (11M rows), Airline Delays (5.8M rows) 데이터셋은 추후 실험

### 다음 단계

- 16GB+ GPU에서 대규모 데이터셋 벤치마크
- VRAM 프로파일 데이터로 VRAMEstimator 회귀 모델 학습
- rmm 풀 전략 비교 실험 (None/Fixed/Managed/Adaptive)

In [ ]:
# 결과 저장 (재현성)
import json

results = {
    "leaderboard": lb.to_dict(orient="records"),
    "comparison": comparison,
    "config": {
        "dataset": "Kaggle Credit Card Fraud",
        "rows": int(df.shape[0]),
        "features": int(df.shape[1]),
        "gpu": "RTX 4060 8GB",
    }
}

output_path = "../../assets/results/benchmark_credit_card.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {output_path}")